In [6]:
import pandas as pd
import pybaseball

In [23]:
from pybaseball import statcast
df = statcast('2025-01-01', '2025-12-31') # pull data all of 2025 season

AssertionError: 

In [32]:
import requests
import pandas as pd

mlb_teams = {
    'ATH': 133, 'PIT': 134, 'SD': 135, 'SEA': 136, 'SF': 137, 'STL': 138,
    'TB': 139, 'TEX': 140, 'TOR': 141, 'MIN': 142, 'PHI': 143, 'ATL': 144,
    'CWS': 145, 'MIA': 146, 'NYY': 147, 'MIL': 158, 'LAA': 108, 'AZ': 109,
    'BAL': 110, 'BOS': 111, 'CHC': 112, 'CIN': 113, 'CLE': 114, 'COL': 115,
    'DET': 116, 'HOU': 117, 'KC': 118, 'LAD': 119, 'WSH': 120, 'NYM': 121
}

def get_mlb_schedule(team_id, season=2025):
    url = f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&teamId={team_id}&season={season}&gameType=R"
    r = requests.get(url).json()
    games = []
    for date in r.get('dates', []):
        for game in date.get('games', []):
            games.append({
                'date': date['date'],
                'home_team': game['teams']['home']['team']['name'],
                'away_team': game['teams']['away']['team']['name'],
                'home_abbr': game['teams']['home']['team'].get('abbreviation', ''),
                'away_abbr': game['teams']['away']['team'].get('abbreviation', ''),
                'game_pk': game['gamePk'],
                'status': game['status']['detailedState'],
            })
    return pd.DataFrame(games)

teams_list = df['home_team'].unique()

results = []
for t in teams_list:
    if t not in mlb_teams:
        print(f"SKIPPED: {t}")
        continue
    try:
        sched = get_mlb_schedule(mlb_teams[t])
        results.append(sched)
        print(f"OK: {t}")
    except Exception as e:
        print(f"FAILED: {t} | {e}")

all_schedules = pd.concat(results, ignore_index=True).drop_duplicates(subset='game_pk')
print(all_schedules.shape)
print(all_schedules.head())

OK: TOR
OK: LAD
OK: SEA
OK: MIL
OK: CHC
OK: NYY
OK: DET
OK: PHI
OK: CLE
OK: BOS
OK: ATL
OK: MIA
OK: WSH
OK: ATH
OK: SF
OK: SD
OK: LAA
OK: BAL
OK: CIN
OK: AZ
OK: TEX
OK: NYM
OK: TB
OK: CWS
OK: PIT
OK: COL
OK: STL
OK: KC
OK: HOU
OK: MIN
(2430, 7)
         date          home_team             away_team home_abbr away_abbr  \
0  2025-03-27  Toronto Blue Jays     Baltimore Orioles                       
1  2025-03-28  Toronto Blue Jays     Baltimore Orioles                       
2  2025-03-29  Toronto Blue Jays     Baltimore Orioles                       
3  2025-03-30  Toronto Blue Jays     Baltimore Orioles                       
4  2025-03-31  Toronto Blue Jays  Washington Nationals                       

   game_pk status  
0   778556  Final  
1   778549  Final  
2   778536  Final  
3   778525  Final  
4   778507  Final  


In [22]:
print(df.columns.tolist())

['pitch_type', 'game_date', 'release_speed', 'release_pos_x', 'release_pos_z', 'player_name', 'batter', 'pitcher', 'events', 'description', 'spin_dir', 'spin_rate_deprecated', 'break_angle_deprecated', 'break_length_deprecated', 'zone', 'des', 'game_type', 'stand', 'p_throws', 'home_team', 'away_team', 'type', 'hit_location', 'bb_type', 'balls', 'strikes', 'game_year', 'pfx_x', 'pfx_z', 'plate_x', 'plate_z', 'on_3b', 'on_2b', 'on_1b', 'outs_when_up', 'inning', 'inning_topbot', 'hc_x', 'hc_y', 'tfs_deprecated', 'tfs_zulu_deprecated', 'umpire', 'sv_id', 'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az', 'sz_top', 'sz_bot', 'hit_distance_sc', 'launch_speed', 'launch_angle', 'effective_speed', 'release_spin_rate', 'release_extension', 'game_pk', 'fielder_2', 'fielder_3', 'fielder_4', 'fielder_5', 'fielder_6', 'fielder_7', 'fielder_8', 'fielder_9', 'release_pos_y', 'estimated_ba_using_speedangle', 'estimated_woba_using_speedangle', 'woba_value', 'woba_denom', 'babip_value', 'iso_value', 'launch_speed_a

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
43,FS,2025-11-01,92.1,-2.02,5.15,"Yamamoto, Yoshinobu",672386,808967,grounded_into_double_play,hit_into_play,...,<NA>,2.33,0.48,0.48,36.9,8.375233,-7.664043,17.979522,48.911447,37.014581
47,CU,2025-11-01,80.3,-1.7,5.45,"Yamamoto, Yoshinobu",672386,808967,NaN,called_strike,...,<NA>,4.85,-1.36,-1.36,48.1,<NA>,<NA>,<NA>,<NA>,<NA>
52,FC,2025-11-01,92.8,-1.91,5.24,"Yamamoto, Yoshinobu",672386,808967,NaN,foul,...,<NA>,2.04,-0.23,-0.23,39.1,14.162806,5.694338,22.383742,47.652657,23.055298
53,FS,2025-11-01,90.9,-1.97,5.26,"Yamamoto, Yoshinobu",680718,808967,walk,blocked_ball,...,<NA>,2.74,0.69,-0.69,40.1,<NA>,<NA>,<NA>,<NA>,<NA>
57,FS,2025-11-01,90.8,-2.02,5.27,"Yamamoto, Yoshinobu",680718,808967,NaN,ball,...,<NA>,2.3,0.23,-0.23,38.0,<NA>,<NA>,<NA>,<NA>,<NA>


In [40]:
count = df[[# 'pitch_type', 
            'home_team', 'description']].value_counts().reset_index()

# apply the filter, don't just store the mask
count_filtered = count[count['count'] > 1]
count_filtered

,home_team,description,count
0,TOR,ball,9340
1,LAD,ball,9309
2,NYY,ball,9307
3,MIL,ball,9220
4,SEA,ball,9002
...,...,...,...
405,CIN,pitchout,2
406,CIN,automatic_strike,2
407,MIA,pitchout,2
408,WSH,pitchout,2


In [17]:
df.shape

(5066, 118)